LightGBM was still on its default parameters. Tuning on val, with clean val alongside, same features as run #13. Early stopping on val picks the number of trees for each try, n_estimators 5000 is only a ceiling.

In [ ]:
import pandas as pd

from prc_fuel.clean import clean_all, clean_training, mask_bad_values
from prc_fuel.evaluate import score
from prc_fuel.features import FEATURES
from prc_fuel.load import load
from prc_fuel.split import split
from prc_fuel.train import train

df = mask_bad_values(load("train"))
fit, val = split(df)
fit = clean_training(fit)
clean_val = clean_all(val)
features = [f for f in FEATURES if f not in ["avg_vertical_rate", "openap_fuel"]]

In [ ]:
params = {"learning_rate": 0.05, "n_estimators": 5000, "verbose": -1}
model = train(fit, features, params, val=val)

model.best_iteration_, score(model, val, features), score(model, clean_val, features)

learning_rate 0.05 with early stopping on val: stops at 233 trees, val 293.4, clean val 161.0 against 295.0 / 163.8 for #13 (defaults, 100 trees). Small gain on both.

In [ ]:
from itertools import product

results = []
for num_leaves, min_child_samples in product([15, 31, 63, 127], [20, 50, 100]):
    params = {
        "learning_rate": 0.05,
        "n_estimators": 5000,
        "num_leaves": num_leaves,
        "min_child_samples": min_child_samples,
        "verbose": -1,
    }
    model = train(fit, features, params, val=val)
    results.append(
        {
            "num_leaves": num_leaves,
            "min_child_samples": min_child_samples,
            "n_trees": model.best_iteration_,
            "val": score(model, val, features),
            "clean_val": score(model, clean_val, features),
        }
    )

pd.DataFrame(results).sort_values("val")

More leaves is better on both val and clean val, 15 and 31 (default) are all at the bottom. min_child_samples is less clear: 20 is best on val with many leaves, 50-100 a bit better on clean val, gaps of 2-3 kg. Best on val 127 / 20: 288.1 / 157.0, about -7 kg on both against #13. 127 is the largest value tried and still the best, so the grid has to go higher.

In [ ]:
from itertools import product

results = []
for num_leaves, min_child_samples in product([127, 255, 511], [20, 50, 100]):
    params = {
        "learning_rate": 0.05,
        "n_estimators": 5000,
        "num_leaves": num_leaves,
        "min_child_samples": min_child_samples,
        "verbose": -1,
    }
    model = train(fit, features, params, val=val)
    results.append(
        {
            "num_leaves": num_leaves,
            "min_child_samples": min_child_samples,
            "n_trees": model.best_iteration_,
            "val": score(model, val, features),
            "clean_val": score(model, clean_val, features),
        }
    )

pd.DataFrame(results).sort_values("val")

Going above 127 leaves doesn't help on val (289.8-292.4 for 255 and 511), only 1-2 kg on clean val, noise level. Plateau reached. Kept num_leaves 127, min_child_samples 20: best on val, and the simpler of the near-ties on clean val.

In [ ]:
results = []
for learning_rate in [0.05, 0.02, 0.01]:
    params = {
        "learning_rate": learning_rate,
        "n_estimators": 5000,
        "num_leaves": 127,
        "min_child_samples": 20,
        "verbose": -1,
    }
    model = train(fit, features, params, val=val)
    results.append(
        {
            "learning_rate": learning_rate,
            "n_trees": model.best_iteration_,
            "val": score(model, val, features),
            "clean_val": score(model, clean_val, features),
        }
    )

pd.DataFrame(results)

A smaller learning rate doesn't help: 0.02 and 0.01 are worse on val (288.9, 289.8), 0.02 is ~1 kg better on clean val, noise level. Number of trees roughly doubles each time (160, 381, 731). Kept 0.05.

Retained: learning_rate 0.05, 160 trees, num_leaves 127, min_child_samples 20.

Run #15 with these as defaults (no early stopping, 160 trees): val 288.1, clean val 157.0, but rank 247.3 against 241.2 for #13. Better on val, worse on rank. val flights are drawn at random from April-August, so they have very close flights in fit (same aircraft, route, week), and bigger trees learn those details. rank is September, new flights, where those details don't carry over. Tuning on val picked the wrong thing, a time-based validation is needed before tuning again.